<a href="https://colab.research.google.com/github/alfredqbit/kannada-mnist/blob/main/sepulvedaADDS_8150_1_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Kannada MNIST Classification: Non-Convolutional Vision Transformer (ViT)
**Author:** A. Sepulveda-Jimenez  
**Organization(s):** National University and QDR Labs  
**Date:** 2026  

---
> &copy; 2026 **A. Sepulveda-Jimenez**. All rights reserved.  
> *Principal Researcher, QDR Labs*

---

### Project Overview
This notebook implements a high-performance **Vision Transformer (ViT)** architecture to classify the Kannada MNIST dataset. Moving away from the standard inductive biases of Convolutional Neural Networks (CNNs), this implementation explores the efficacy of **Global Self-Attention** and **Patch Embeddings** for handwritten digit recognition.

**Key Technical Specifications:**
* **Architecture:** Pure Vision Transformer (ViT) with Multi-Head Attention.
* **Optimization:** PyTorch Lightning pipeline utilizing **16-bit Mixed Precision (AMP)**.
* **Hardware Support:** Fully optimized for seamless execution on **Kaggle GPU (P100/T4)** and **TPU v3-8** accelerators.
* **Objective:** Achieve state-of-the-art accuracy on $28 \times 28$ grayscale images using non-convolutional spatial feature extraction.

---

Cell 1: Install and Import Dependencies

In [ ]:
# If running in Kaggle or Colab, ensure lightning is installed

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
try:
    import pytorch-lightning as pl
except ImportError:
    print("pytorch-lightning not found. Installing...")
    !pip install pytorch-lightning -q
    import pytorch_lightning as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

Cell 2: Optimized Data Pipeline (LightningDataModule)

In [ ]:
class KannadaDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

class KannadaDataModule(pl.LightningDataModule):
    def __init__(self, file_path: str, batch_size: int = 128, num_workers: int = 4):
        super().__init__()
        self.file_path = file_path
        self.batch_size = batch_size
        self.num_workers = num_workers # Parallelize data loading

    def setup(self, stage=None):
        try:
            data = pd.read_csv(self.file_path)
        except FileNotFoundError:
            print(f"File not found at {self.file_path}. Adjust path if needed.")
            return

        X = data.drop('label', axis=1).values
        y = data['label'].values

        # Normalize and Reshape
        X = X.astype(np.float32) / 255.0
        X = X.reshape(-1, 1, 28, 28)

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        self.train_dataset = KannadaDataset(X_train, y_train)
        self.val_dataset = KannadaDataset(X_val, y_val)

    def train_dataloader(self):
        # pin_memory speeds up CPU to GPU/TPU data transfer
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )

Cell 3: Optimized ViT Model (LightningModule)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=1, patch_size=7, emb_size=64):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, emb_size, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class LitKannadaViT(pl.LightningModule):
    def __init__(self, in_channels=1, patch_size=7, emb_size=64, img_size=28, num_classes=10, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr

        self.patch_embed = PatchEmbedding(in_channels, patch_size, emb_size)
        num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, emb_size))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_size, nhead=4, dim_feedforward=128, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.mlp_head = nn.Sequential(
            nn.Linear(emb_size, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
            nn.Softmax(dim=1)
        )

        self.criterion = nn.NLLLoss()

    def forward(self, x):
        b = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embed

        x = self.transformer(x)
        x = x[:, 0]
        return self.mlp_head(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(torch.log(outputs + 1e-9), labels)

        # Log metrics to TensorBoard/Weights & Biases
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(torch.log(outputs + 1e-9), labels)

        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

Cell 4: Hardware-Optimized Execution
By setting accelerator='auto', PyTorch Lightning will automatically detect if you are running this on a Kaggle TPU, a Kaggle GPU, or a standard CPU, and route the tensor operations accordingly without requiring manual .to(device) calls or TPU xla wrappers.

In [ ]:
# Initialize DataModule
data_module = KannadaDataModule(
    file_path='/content/data/train.csv', # Updated to the correct local path
    batch_size=256, # Can often use larger batches with AMP
    num_workers=4   # Adjust based on Kaggle/local CPU cores
)

# Initialize Model
model = LitKannadaViT(lr=0.001)

# Configure the Trainer
trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto', # Automatically detects GPU, TPU, or CPU
    devices='auto',     # Uses all available accelerators
    precision='bf16-true', # TPU-compatible precision format
    enable_progress_bar=True
)

# Execute Pipeline
print("Starting hardware-accelerated training pipeline...")
trainer.fit(model, datamodule=data_module)

Cell 5: Generate Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import torch
import os
from sklearn.metrics import confusion_matrix, classification_report

# Create figures directory
os.makedirs('figures', exist_ok=True)

# Ensure data_module is setup to access datasets
data_module.setup()

# Extract datasets for visualization
train_images = data_module.train_dataset.images.numpy()
train_labels = data_module.train_dataset.labels.numpy()
val_images = data_module.val_dataset.images.numpy()
val_labels = data_module.val_dataset.labels.numpy()

# 1. Sample Images Plot
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Kannada MNIST Digits', fontsize=16)
for i, ax in enumerate(axes.flatten()):
    ax.imshow(train_images[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {train_labels[i]}")
    ax.axis('off')
plt.tight_layout()
plt.savefig('figures/sample_images.png', bbox_inches='tight')
plt.show()

# 2. Class Distribution Balance Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x=train_labels, ax=axes[0], hue=train_labels, palette='viridis', legend=False)
axes[0].set_title('Train Dataset Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

sns.countplot(x=val_labels, ax=axes[1], hue=val_labels, palette='viridis', legend=False)
axes[1].set_title('Validation Dataset Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('figures/class_distribution.png', bbox_inches='tight')
plt.show()

# 3. Pixel Distribution Plot
plt.figure(figsize=(8, 5))
# Sample pixels for faster plotting
sns.histplot(train_images.flatten()[::100], bins=50, kde=False, color='teal')
plt.title('Pixel Intensity Distribution (Sampled)')
plt.xlabel('Pixel Value (Normalized)')
plt.ylabel('Frequency')
plt.savefig('figures/pixel_distribution.png', bbox_inches='tight')
plt.show()

# --- Inference on Validation Set ---
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

val_loader = data_module.val_dataloader()
all_preds = []
all_labels = []

print("Running inference on validation set...")
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 4. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('figures/confusion_matrix.png', bbox_inches='tight')
plt.show()

# 5. Accuracy Metrics & Per Class Accuracy
report = classification_report(all_labels, all_preds, output_dict=True)
print("Classification Report:\n", classification_report(all_labels, all_preds))

per_class_acc = [report[str(i)]['recall'] for i in range(10)]
plt.figure(figsize=(10, 5))
sns.barplot(x=list(range(10)), y=per_class_acc, hue=list(range(10)), palette='magma', legend=False)
plt.title('Per-Class Accuracy (Recall)')
plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
for i, acc in enumerate(per_class_acc):
    plt.text(i, acc + 0.02, f"{acc:.2f}", ha='center')
plt.savefig('figures/per_class_accuracy.png', bbox_inches='tight')
plt.show()

# 6. Misclassification Plot
misclassified_idx = np.where(all_preds != all_labels)[0]
if len(misclassified_idx) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('Sample Misclassifications', fontsize=16)
    for i, ax in enumerate(axes.flatten()):
        if i < len(misclassified_idx):
            idx = misclassified_idx[i]
            img = val_images[idx].reshape(28, 28)
            true_label = all_labels[idx]
            pred_label = all_preds[idx]
            ax.imshow(img, cmap='gray')
            ax.set_title(f"True: {true_label} | Pred: {pred_label}", color='red')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.savefig('figures/misclassifications.png', bbox_inches='tight')
    plt.show()
else:
    print("No misclassifications found!")

# 7. Training Curves
log_dir = 'lightning_logs'
if os.path.exists(log_dir):
    versions = [d for d in os.listdir(log_dir) if d.startswith('version_')]
    if versions:
        # Sort by version number to get the latest
        latest_version = sorted(versions, key=lambda x: int(x.split('_')[1]))[-1]
        metrics_file = os.path.join(log_dir, latest_version, 'metrics.csv')
        if os.path.exists(metrics_file):
            metrics = pd.read_csv(metrics_file)
            # Forward fill to handle different logging intervals (e.g., train vs val)
            metrics.ffill(inplace=True)

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            if 'train_loss' in metrics.columns and 'val_loss' in metrics.columns:
                axes[0].plot(metrics['step'], metrics['train_loss'], label='Train Loss', alpha=0.7)
                axes[0].plot(metrics['step'], metrics['val_loss'], label='Val Loss', linewidth=2)
                axes[0].set_title('Training and Validation Loss')
                axes[0].set_xlabel('Steps')
                axes[0].set_ylabel('Loss')
                axes[0].legend()

            if 'train_acc' in metrics.columns and 'val_acc' in metrics.columns:
                axes[1].plot(metrics['step'], metrics['train_acc'], label='Train Acc', alpha=0.7)
                axes[1].plot(metrics['step'], metrics['val_acc'], label='Val Acc', linewidth=2)
                axes[1].set_title('Training and Validation Accuracy')
                axes[1].set_xlabel('Steps')
                axes[1].set_ylabel('Accuracy')
                axes[1].legend()

            plt.tight_layout()
            plt.savefig('figures/training_curves.png', bbox_inches='tight')
            plt.show()
        else:
            print(f"Metrics file not found in {latest_version}.")
else:
    print("Lightning logs not found. Training curves cannot be plotted.")

Cell 6: Generate Kaggle Submission

In [ ]:
import os

# 1. Load the Kaggle test dataset
test_file_path = '/content/data/test.csv'

try:
    test_data = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"File not found at {test_file_path}. Please adjust the path.")
    # Fallback for local testing
    # test_data = pd.read_csv('test.csv')

# 2. Extract IDs and Preprocess the pixel data
test_ids = test_data['id'].values
X_test_submission = test_data.drop('id', axis=1).values

# Normalize and Reshape (matching the training preprocessing exactly)
X_test_submission = X_test_submission.astype(np.float32) / 255.0
X_test_submission = X_test_submission.reshape(-1, 1, 28, 28)

# Convert to PyTorch tensors and create a DataLoader
test_tensor = torch.tensor(X_test_submission)
# TensorDataset wraps the tensor in a tuple, which is standard for DataLoaders
submission_dataset = torch.utils.data.TensorDataset(test_tensor)
submission_loader = DataLoader(submission_dataset, batch_size=256, shuffle=False)

# 3. Run Inference
# Ensure the model is in evaluation mode and on the correct hardware device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

predictions = []

print("Generating predictions...")
with torch.no_grad():
    for batch in submission_loader:
        # batch[0] contains the images
        images = batch[0].to(device)

        # Forward pass
        outputs = model(images)

        # Get the index of the highest probability
        preds = torch.argmax(outputs, dim=1)

        # Move predictions back to CPU and convert to numpy for pandas
        predictions.extend(preds.cpu().numpy())

# 4. Format and Export the Submission
submission_df = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

submission_df.to_csv('submission.csv', index=False)
print(f"Submission saved to {os.getcwd()}/submission.csv")
print(submission_df.head())